# Khmer TTS — Step 1: Text data & preprocessing
**Goal:** understand the dataset, pick ONE speaker, clean the Khmer text, and save a clean `speaker_manifest.csv` that Member 2 (audio) will use.

Steps in this notebook
1. Load the metadata files
2. Dataset statistics (speakers, topics, subtopics)
3. Choose the single speaker
4. Look at the raw text (find the problems *before* cleaning)
5. Clean the text (Unicode, punctuation, characters)
6. Link each sentence to its `.wav` file name
7. Save the manifest

## Before you start: conda environment
Run these in **Anaconda Prompt / terminal** (not inside the notebook), once:

```bash
conda create -n khmer-tts python=3.10 -y
conda activate khmer-tts
conda install -c conda-forge pandas numpy matplotlib scikit-learn tqdm pyyaml librosa soundfile ffmpeg jupyterlab ipykernel -y
pip install pyloudnorm
python -m pip install ipykernel jupyterlab
python -m ipykernel install --user --name khmer-tts --display-name "Python (khmer-tts)"
jupyter lab
```
conda install -c conda-forge pandas matplotlib soundfile librosa scikit-learn tqdm ipykernel jupyterlab
Then open this notebook and choose the kernel **Python (khmer-tts)**.

**Later steps (do not install yet):**
- Step 3, VITS training with an NVIDIA GPU: `pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121` then `pip install coqui-tts`. No GPU? Train on Google Colab or Kaggle.
- Step 4: `pip install gradio jiwer`
- If `coqui-tts` conflicts with your packages, create a separate env (`conda create -n khmer-train python=3.10`) for training.

In [ ]:
# Check the environment (run this cell first)
import sys, importlib
print("Python:", sys.version.split()[0])
for m in ["pandas","numpy","matplotlib","soundfile","librosa","sklearn","tqdm"]:
 try: print(f"OK {m} {importlib.import_module(m).__version__}")
 except Exception as e: print(f"MISSING {m} -> conda install -c conda-forge {m}")

In [ ]:
import soundfile as sf

print("SoundFile version:", sf.__version__)
print("Installation successful!")

## 0. Setup


In [ ]:
import re, unicodedata, collections
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

META_DIR = Path(r"C:\Users\MSI\Pictures\NLPproject\dataset") # <- change me
WAV_DIR = Path(r"C:\Users\MSI\Pictures\NLPproject\dataset\data") # <- change me (folder with burials/, dining/, ...)
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)

## 1. Load the metadata
Two tables: one row per **sentence** (`metadata.csv`) and one row per **speaker** (`speaker_metadata.csv`).

In [ ]:
meta = pd.read_csv(META_DIR/"metadata.csv")
spk = pd.read_csv(META_DIR/"speaker_metadata.csv")
print(meta.shape, spk.shape)
meta.head()

## 2. Dataset statistics


In [ ]:
print("Missing values:\n", meta.isna().sum(), "\n")
print("Sentences per speaker:")
per_spk = meta.groupby("Speaker ID").size().rename("n_sentences").reset_index().merge(spk, on="Speaker ID")
per_spk.sort_values("n_sentences", ascending=False)

**What you should notice:** 6 speakers have thousands of sentences, but 4 speakers (`*-0001`) have only 16 each — they are too small for training. So the real usable speakers are **6**, not 8. Mention this in your report.

In [ ]:
# Topic and subtopic distribution (all speakers)
topic_counts = meta["Topic"].value_counts()
display(topic_counts)
ax = topic_counts.sort_values().plot.barh(figsize=(7,4), title="Sentences per topic (all speakers)")
plt.tight_layout(); plt.savefig(OUT_DIR/"topic_distribution.png", dpi=150); plt.show()

print("Number of subtopics:", meta["Subtopic"].nunique())
meta.groupby(["Topic","Subtopic"]).size().sort_values(ascending=False).head(15)

## 3. Choose the single speaker
For TTS, one voice = consistent output. Criteria: (a) a lot of data, (b) all 11 topics covered, (c) clear/standard accent.

`f-adt2-0002` (female, 35, Phnom Penh) has the full script (11,375 sentences, 11 topics)

In [ ]:
SPEAKER = "f-adt2-0002"
sp = meta[meta["Speaker ID"] == SPEAKER].copy()
print(len(sp), "rows,", sp["Topic"].nunique(), "topics,", sp["Subtopic"].nunique(), "subtopics")

# duplicated sentence IDs? (they would collide with the same wav file name)
dups = sp[sp.duplicated("Sentence ID", keep=False)]
print("Rows sharing a Sentence ID:", len(dups))
# Same ID but DIFFERENT text -> we can't know which sentence the wav contains. Drop them.
sp = sp.drop_duplicates("Sentence ID", keep=False).copy()
print("After dropping ambiguous IDs:", len(sp))
dups[["Sentence ID","Sentences"]]

## 4. Look at the raw text first
Khmer text has hidden problems. Let's measure them before deciding how to clean.

In [ ]:
def non_khmer_chars(texts):
 c = collections.Counter(ch for t in texts for ch in t if not ("\u1780" <= ch <= "\u17FF"))
 return c

for ch, n in non_khmer_chars(sp["Sentences"]).most_common(25):
 print(f"{ch!r:10} U+{ord(ch):04X} {unicodedata.name(ch, '?'):30} {n}")

**Findings (typical for this dataset):**
- `U+200B` ZERO WIDTH SPACE is used a lot — it is *invisible* but marks word boundaries. Khmer has no spaces between words, so writers add it.
- Mixed quote styles: `"` `“ ”` `« »`
- A few Latin letters, digits, `\n` and non-breaking spaces (`U+00A0`).
- Khmer punctuation: `។` (khan, full stop), `៕` (bariyoosan), `ៗ` (repeat mark).

In [ ]:
# See the problems in real sentences
for pat, name in [("\u200b","zero-width space"), (r"[A-Za-z]","latin letters"), (r"[0-9]","digits"), ("\n","newline"), (r"[«»“”\"]","quotes")]:
 ex = sp[sp["Sentences"].str.contains(pat)]["Sentences"].head(1).tolist()
 print(f"--- {name}: {sp['Sentences'].str.contains(pat).sum()} sentences"); print(ex)

## 5. Clean the text
*methodology*:

| Problem | Decision | Why |
|---|---|---|
| Unicode forms | `NFC` normalize | same-looking text can have different code points |
| Khmer coeng/vowel order | keep as is | reordering needs a Khmer-specific library; NFC is enough for a first version |
| `U+200B` zero-width space | **remove** | invisible, the model can't hear it. Real spaces remain as pauses |
| `\n`, `\xa0`, repeated spaces | → single space | |
| Quotes `"“”«»` and brackets | remove | not spoken |
| `។ ៕ ? ! ,` | keep `។` `?` `!` `,` ; map `៕` → `។` | punctuation gives the model pause/intonation cues |
| `ៗ` (repeat mark) | keep for now | it is spoken as repetition; decide with the team |
| Latin letters / digits | **flag**, don't delete | deleting changes the meaning; later either drop those rows or spell out numbers |

In [ ]:
KEEP_PUNCT = "។?!,"
ZW = dict.fromkeys(map(ord, "\u200b\u200c\u200d\ufeff"), None)

def clean_khmer(t: str) -> str:
 t = unicodedata.normalize("NFC", t)
 t = t.translate(ZW) # remove zero-width chars
 t = t.replace("៕", "។")
 t = re.sub(r"[\"“”«»‘’()\[\]{}]", " ", t) # drop quotes/brackets
 t = t.replace("–", " ").replace("-", " ").replace(":", " ").replace("៖", " ").replace("/", " ")
 t = re.sub(r"[\s\u00a0]+", " ", t) # all whitespace -> one space
 t = re.sub(r"\s+([។?!,])", r"\1", t) # no space before punctuation
 return t.strip()

sp["text"] = sp["Sentences"].map(clean_khmer)
sp["has_latin"] = sp["text"].str.contains(r"[A-Za-z]")
sp["has_digit"] = sp["text"].str.contains(r"[0-9\u17E0-\u17E9]") # Latin + Khmer digits
sp["n_chars"] = sp["text"].str.len()

before, after = sp.loc[sp["Sentences"].str.contains("\u200b")].iloc[0], None
print("BEFORE:", repr(before["Sentences"])); print("AFTER :", repr(before["text"]))
print("Latin:", sp.has_latin.sum(), "| digits:", sp.has_digit.sum())

In [ ]:
# Which characters remain? Everything outside Khmer should now be tiny.
for ch, n in non_khmer_chars(sp["text"]).most_common(20):
 print(f"{ch!r:6} U+{ord(ch):04X} {n}")

sp["n_chars"].describe()

In [ ]:
# Drop clearly unusable rows: too short/long (VITS likes sentences ~ 1-15 s => roughly 10-250 chars), or with Latin letters/digits
MIN_CH, MAX_CH = 10, 250
keep = (sp.n_chars.between(MIN_CH, MAX_CH)) & (~sp.has_latin) & (~sp.has_digit)
print("kept", keep.sum(), "of", len(sp), "| removed", (~keep).sum())
sp_clean = sp[keep].copy()
sp_clean["n_chars"].hist(bins=40, figsize=(6,3)); plt.title("Characters per sentence (clean)"); plt.show()

## 6. Link each sentence to its audio file
 wav names look like `f-adt2-0002_khm_wedding_12_0001_0004.wav`.
That is exactly: **`{Speaker ID}_khm_{Sentence ID}.wav`** and it sits inside the topic folder (`wedding/`).

note about `_f-adt1-0002_khm_wedding_17_0001_0009.wav (unsupported)`: a **leading underscore** (or other odd prefix) breaks a strict name match. Our parser ignores leading junk characters so those files are not lost. The folder names also differ from the Topic column (`giving_gift` vs `Giving Gift`, `appliances` vs `Traditional Home Appliances`), so we match by file name only, not by folder.

In [ ]:
# Index every wav on disk: key = Sentence ID (+ speaker)
FILE_RE = re.compile(r"^[_\s.\-]*(?P<spk>[fm]-adt\d-\d{4})_khm_(?P<sid>.+)\.wav$", re.I)

def index_wavs(root: Path):
    rows, bad = [], []
    for p in root.rglob("*"):
        if p.suffix.lower() != ".wav": continue
        m = FILE_RE.match(p.name)
        (rows if m else bad).append((m["spk"], m["sid"], p) if m else p)
    return pd.DataFrame(rows, columns=["Speaker ID","Sentence ID","wav_path"]), bad

if WAV_DIR.exists():
    wavs, unparsed = index_wavs(WAV_DIR)
    print(len(wavs), "wav files parsed;", len(unparsed), "could not be parsed")
    print(unparsed[:5])
else:
    print("WAV_DIR not found yet - set it and re-run. Skipping the file check.")
    wavs = None

In [ ]:
if wavs is not None:
 wavs_spk = wavs[wavs["Speaker ID"] == SPEAKER].drop_duplicates("Sentence ID")
 manifest = sp_clean.merge(wavs_spk, on=["Speaker ID","Sentence ID"], how="left")
 print("Sentences with NO wav:", manifest.wav_path.isna().sum())
 print("Wavs with NO sentence:", len(set(wavs_spk["Sentence ID"]) - set(sp_clean["Sentence ID"])))
else:
 manifest = sp_clean.assign(wav_path=pd.NA)
manifest[["Sentence ID","Topic","Subtopic","text","n_chars","wav_path"]].head()

## 7. Save the outputs
`speaker_manifest.csv` for audio processing. The stats table goes into the report.

In [ ]:
manifest = manifest[manifest.wav_path.notna()] if wavs is not None else manifest
cols = ["Speaker ID","Topic","Subtopic","Paragraph ID","Sentence ID","text","n_chars","wav_path"]
manifest[cols].to_csv(OUT_DIR/"speaker_manifest.csv", index=False)
per_spk.to_csv(OUT_DIR/"speaker_stats.csv", index=False)
manifest.groupby(["Topic","Subtopic"]).size().rename("n").reset_index().to_csv(OUT_DIR/"topic_stats.csv", index=False)
print("saved", len(manifest), "rows to", OUT_DIR)

## Duration statistics (Not important for the model, but useful for the report)
Total hours, average clip length and SD (the proposal reports 134.6 h / 8.54 s / 3.37 s for the whole dataset).

In [ ]:
FILE_RE = re.compile(r"^(?P<spk>[fm]-adt\d-\d{4})_khm_(?P<sid>.+)\.wav$", re.I)

def index_wavs(root: Path):
    rows, bad = [], []
    for p in root.rglob("*"):
        if p.suffix.lower() != ".wav":
            continue
        # skip macOS junk: "._name.wav" files and anything inside __MACOSX
        if p.name.startswith("._") or "__MACOSX" in p.parts:
            continue
        m = FILE_RE.match(p.name)
        if m:
            rows.append((m["spk"], m["sid"], p))
        else:
            bad.append(p)
    return pd.DataFrame(rows, columns=["Speaker ID", "Sentence ID", "wav_path"]), bad

if WAV_DIR.exists():
    wavs, unparsed = index_wavs(WAV_DIR)
    print(len(wavs), "wav files parsed;", len(unparsed), "could not be parsed")
    print(unparsed[:5])
else:
    print("WAV_DIR not found yet - set it and re-run. Skipping the file check.")
    wavs = None

In [ ]:
import soundfile as sf

def safe_duration(p):
    try:
        return sf.info(p).duration
    except Exception:
        return None          # corrupt / unreadable file

if wavs is not None and len(manifest):
    manifest["duration"] = [safe_duration(p) for p in manifest["wav_path"]]

    bad = manifest[manifest["duration"].isna()]
    print("Unreadable files:", len(bad))
    if len(bad):
        display(bad[["Sentence ID", "wav_path"]].head())
        bad[["Sentence ID", "wav_path"]].to_csv(OUT_DIR / "corrupt_wavs.csv", index=False)

    manifest = manifest.dropna(subset=["duration"]).copy()

    print(f"Total: {manifest.duration.sum()/3600:.2f} h | "
          f"mean {manifest.duration.mean():.2f}s | SD {manifest.duration.std():.2f}s")
    display(manifest.groupby("Topic").duration.sum().div(3600).round(2))